## Problem 1
### Part 1

In [3]:
import sympy as sp
from sympy import symbols, pprint, Function, simplify, Derivative, nsimplify
from sympy import sin, cos, asin, acos, pi, diff, eye
from sympy import Matrix, latex, BlockMatrix, lambdify
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sympy import init_printing
init_printing() 

# Define Symbols
t  = symbols("t",   real=True)
l = symbols("l",  positive=True)   # Length of Link, assume Links are same length
#l2 = symbols("l2",  positive=True)   # Length of Link 2
#l3 = symbols("l3",  positive=True)   # Lrngth of link 3
k = symbols("k", positive=True) # Drag Coefficient
# Config Variables
x     = Function('x',     real=True)(t)
y     = Function('y',     real=True)(t)
theta= Function('theta', real=True)(t)
a1= Function('a1',real=True)(t)
a2= Function('a2',real=True)(t)


In [4]:
# Defining Various Helper Functions
def cross_matrix(vec):
    
    return Matrix([[0, -vec[2,0], vec[1,0]],
                      [vec[2,0], 0, -vec[0,0]],
                      [-vec[1,0], vec[0,0], 0]])

def uncross_matrix(mat):
    
    return Matrix([[mat[2,1]],
                      [mat[0,2]], 
                      [-mat[0,1]]])

def twist2vec(twist):
    m = Matrix.zeros(6,1)
    
    m[0:3,0] = uncross_matrix(twist[0:3,0:3])
    m[3:,0] = twist[0:3,3]
    
    return(m)
    
def R_z(theta):
    """
    Function to return an arbitrary transformation matrix 
    This is for sympy symbolic calculation
    """
    return Matrix([[cos(theta), -sin(theta), 0], 
                   [sin(theta), cos(theta), 0],
                   [0, 0, 1]])

def R_y(theta):
    """
    Function to return an arbitrary transformation matrix 
    This is for sympy symbolic calculation
    """
    return Matrix([[cos(theta),0, sin(theta)], 
                   [0, 1, 0],
                   [-sin(theta), 0, cos(theta)]])

def R_x(theta):
    """
    Function to return an arbitrary transformation matrix 
    This is for sympy symbolic calculation
    """
    return Matrix([[1, 0, 0],
                   [0, cos(theta), -sin(theta)], 
                   [0, sin(theta), cos(theta)]])
def T(R, p):
    m = Matrix.zeros(4,4)
    m[0:3, 0:3] = R
    m[:3, 3] = p
    m[3, 3] = 1
    return m


In [5]:
# Transform to Joint 1, then Translate to COM 1
Tjoint1   = T(R_z(theta), Matrix([[x],[y],[0]]))
TJ1toCOM1 = T(eye(3), Matrix([[l1/2],[0],[0]]))
T1 = simplify(Tjoint1 @ TJ1toCOM1) # World Frame to COM 1

# Rotate alpha 1 about JOint 1, then Translate to COM
TJ1a1= T(R_z(a1), Matrix([[0],[0],[0]]))
TJ1toCOM2 = T(eye(3), Matrix([[-l2/2],[0],[0]]))
T2 = simplify(Tjoint1 @ TJ1a1 @ TJ1toCOM2) # World Frame to COM 2

# Translate to Joint 2, and Rotate alpha 2, then translate to COM 3
TJ1toJ2= T(R_z(a2), Matrix([[l1],[0],[0]]))
TJ2toCOM3 = T(eye(3), Matrix([[l3/2],[0],[0]]))
T3 = simplify(Tjoint1 @ TJ1toJ2 @ TJ2toCOM3) # World Frame to COM 3

NameError: name 'l1' is not defined

In [ ]:
T1

In [ ]:
T2

In [ ]:
T3

### Part 2 Compute Body Twists for each COM

In [ ]:
# Body Twist For Each COM
bodytwist_COM1 = simplify(T1.inv() @ diff(T1, t))
bodytwist_COM1 = twist2vec(bodytwist_COM1) # Vectorized Bodytwist for COM 1
bodytwist_COM2 = simplify(T2.inv() @ diff(T2, t))
bodytwist_COM2 = twist2vec(bodytwist_COM2) # Vectorized Bodytwist for COM 2
bodytwist_COM3 = simplify(T3.inv() @ diff(T3, t))
bodytwist_COM3 = twist2vec(bodytwist_COM3) # Vectorized Bodytwist for COM 3

In [ ]:
bodytwist_COM1

In [ ]:
bodytwist_COM2

In [ ]:
bodytwist_COM3

### Part 3 Compute Jacobian

In [ ]:
# Define q_dot
xd     = diff(x, t)
yd     = diff(y, t)
thetad = diff(theta, t)
a1d    = diff(a1, t)
a2d    = diff(a2, t)

q_dot = Matrix([xd, yd, thetad, a1d, a2d])

# Body Twist Jacobians
J1 = bodytwist_COM1.jacobian(q_dot)
J2 = bodytwist_COM2.jacobian(q_dot)
J3 = bodytwist_COM3.jacobian(q_dot)

In [ ]:
J1

In [ ]:
J2

In [ ]:
J3

### Part 4 Body Wrench

In [ ]:
# Viscous Force Matrix
def BodyWrench(bodytwist):
    B = Matrix([
    [0, 0,        0, 0,   0,   0],
    [0, 0,        0, 0,   0,   0],
    [0, 0, 2*k*l**3, 0,   0,   0],
    [0, 0,        0, k*l, 0,   0],
    [0, 0,        0, 0, 2*k*l, 0],
    [0, 0,        0, 0,   0,   0]
])
    return -B @ bodytwist

# Call BodyWrenches
F1 = BodyWrench(bodytwist_COM1)
F2 = BodyWrench(bodytwist_COM2)
F3 = BodyWrench(bodytwist_COM3)

In [ ]:
F1

In [ ]:
F2

In [ ]:
F3

### Part 5